# Member 2 — CNN from scratch / Lab 2
Phase 1 فقط. المدخل: مخرجات Member 1 بالإصدار الجديد. مهمتان: empty/animal على الصورة الكاملة وspecies على human-box crops. الكود مباشر ولا يحتاج Drive.
شغّل الخلايا بالترتيب؛ اختيار النماذج على validation فقط. لا تعدّل التجارب بعد مشاهدة Test.

## 1. البيئة واستلام البيانات

In [ ]:
%pip -q install numpy pandas scipy scikit-learn matplotlib Pillow opencv-python-headless wandb 'transformers==4.49.0'


In [ ]:
from pathlib import Path
import os,json,hashlib,sys,subprocess,platform,random
import numpy as np,pandas as pd
from IPython.display import display,Markdown
ROOT=Path.cwd() if Path.cwd().name=='CameraTrap_Phase1_Team' else Path.cwd()/'CameraTrap_Phase1_Team'
ROOT.mkdir(exist_ok=True);os.chdir(ROOT)
for name in ['handoff','runs','review','submission','docs']:Path(name).mkdir(exist_ok=True)
def read_json(p):return json.loads(Path(p).read_text())
def write_json(p,v):
    Path(p).parent.mkdir(parents=True,exist_ok=True)
    Path(p).write_text(json.dumps(v,indent=2,ensure_ascii=False,allow_nan=False))
def digest(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def environment(member):
    Path(f'handoff/requirements_member{member}.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
    write_json(f'handoff/runtime_member{member}.json',{'python':sys.version,'platform':platform.platform()})
print('Workspace:',ROOT)


In [ ]:
# ارفع الحزمة من Files في Colab، ثم اكتب مسارها هنا. اترك القائمة فارغة إذا الملفات موجودة.
ARCHIVES = []  # مثال: ['/content/Member1_Results.zip']
import zipfile
for path in ARCHIVES:
    with zipfile.ZipFile(path) as archive:
        # Validate all paths and conflicts BEFORE extracting; never silently overwrite another release.
        for item in archive.infolist():
            target=(ROOT/item.filename).resolve()
            assert target.is_relative_to(ROOT.resolve()), 'Unsafe archive path'
            if target.is_file() and not item.is_dir():
                assert hashlib.sha256(archive.read(item)).hexdigest()==digest(target), f'Conflict: {item.filename}'
        archive.extractall(ROOT)
print('Imported:',len(ARCHIVES),'archives')


In [ ]:
release=read_json('handoff/dataset_release.json')
assert release['schema']==2 and release['audit_complete'], 'استخدم إصدار Member 1 الجديد بعد اكتمال التدقيق'
for p,h in release['files'].items():assert Path(p).is_file() and digest(p)==h, f'Changed/missing: {p}'
DATA=Path(release['data_dir']);rows=read_json(DATA/'manifest.json')
assert rows and all(Path(r['path']).exists() for r in rows)
DATA_HASH=digest(DATA/'manifest.json')
CFG=read_json('handoff/data_config.json')
assert CFG['preprocessing_sha256']=='829fef7a32c1157ffe4da29da42026105afc0266ed647e562127bcc416697937'
print('Frames:',len(rows),'| split sizes:',pd.Series([r['split'] for r in rows]).value_counts().to_dict())


In [ ]:
import cv2, numpy as np, torch
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader
class Preprocessor:
    """Lab 1 cell 72: prepare uint8 RGB, augment, finalize and CHW tensor."""
    def __init__(self,size):
        self.size=size
        self.mean=np.array([.485,.456,.406],np.float32)
        self.std=np.array([.229,.224,.225],np.float32)
    def prepare(self,bgr):
        rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB);h,w=rgb.shape[:2]
        interp=cv2.INTER_AREA if h>self.size or w>self.size else cv2.INTER_LINEAR
        return cv2.resize(rgb,(self.size,self.size),interpolation=interp)
    def finalize(self,rgb):return (rgb.astype(np.float32)/255.-self.mean)/self.std

def load_rgb_sample(row):
    im=cv2.imread(row['path'])
    if im is None:raise ValueError('Unreadable image: '+row['path'])
    if row.get('crop') is not None:
        x1,y1,x2,y2=row['crop'];h,w=im.shape[:2];dx=(x2-x1)*.05;dy=(y2-y1)*.05
        left,top=max(0,int((x1-dx)*w)),max(0,int((y1-dy)*h))
        right,bottom=min(w,max(left+1,int((x2+dx)*w))),min(h,max(top+1,int((y2+dy)*h)))
        im=im[top:bottom,left:right].copy()
    return im

class FrameDataset(torch.utils.data.Dataset):
    """Lab 1 cell 104 adapted to frame and human-box crop tasks."""
    def __init__(self,samples,size,train=False):
        self.samples=samples;self.pre=Preprocessor(size)
        self.tf=transforms.Compose([transforms.RandomHorizontalFlip(),
                  transforms.ColorJitter(brightness=.15,contrast=.15)]) if train else None
    def __len__(self):return len(self.samples)
    def __getitem__(self,i):
        row=self.samples[i];img=self.pre.prepare(load_rgb_sample(row))
        if self.tf:img=np.array(self.tf(Image.fromarray(img)))
        x=torch.from_numpy(np.ascontiguousarray(self.pre.finalize(img))).permute(2,0,1).float()
        return x,int(row['label']),i

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def artifact_context(cfg, task):
    return dict(task=task, profile=cfg['profile'], image_size=cfg['image_size'],
                seed=cfg['seed'], manifest_sha256=digest(Path(cfg['data_dir'])/'manifest.json'),
                split_policy='camera-disjoint train / val_select / val_calibrate / test',
                preprocessing_sha256=cfg['preprocessing_sha256'],
                training_config={k:cfg[k] for k in ['optimizer','backbone','batch_size','epochs','patience','warmup_epochs',
                    'lr_scratch','lr_frozen','lr_full','weight_decay','label_smoothing','class_weighted_loss']})

def samples_for_task(rows,task,classes):
    samples=[]
    for r in rows:
        if task=='triage':
            samples.append(dict(r,label=r['animal'],sample_id=r['id'],crop=None))
        else:
            for i,b in enumerate(r['boxes']):
                samples.append(dict(r,label=classes.index(b['species']) if b['species'] in classes else -1,
                                    sample_id=f'{r["id"]}_{i}',crop=b['xyxy'],species=b['species']))
    return samples

## 2. إعدادات مشتركة وتجارب W&B
الإعدادات من Member 1؛ أي تعديل تجريبي يُثبت قبل الاختبار. استخدم مجلد نتائج جديدًا عند تغيير الإعدادات. offline يحفظ السجلات محليًا؛ زامنها قبل التسليم.

In [ ]:
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE)
if DEVICE=='cpu':print('CPU بطيء للتدريب الكامل؛ اختر GPU في Colab')
WANDB_MODE='offline'  # online لحساب W&B؛ لا تكتب المفتاح في الكود
import wandb
if WANDB_MODE=='online':wandb.login()
REGIMES=['scratch']
OUT=Path('runs/member2');OUT.mkdir(parents=True,exist_ok=True)
CFG['output_dir']=str(OUT)
assert CFG['optimizer']=='Adam'
write_json(OUT/'config.json',CFG)
environment(2)


## 3. النموذج وforward — Lab 2 cell 39
ConvNN: ثلاث كتل Conv/BatchNorm/ReLU ثم pooling وطبقات fully connected. الأبعاد صريحة بدل LazyLinear، مع dropout 0.2.

In [ ]:
from torch import nn

class ConvNN(nn.Module):
    """Lab 2 cell 39 architecture; debug helpers removed, dimensions explicit."""
    def __init__(self,num_classes,size=224,num_filters=16,num_neurons=100):
        super().__init__()
        def block(a,b):return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU())
        self.block_1=block(3,num_filters)
        self.block_2=block(num_filters,2*num_filters)
        self.block_3=block(2*num_filters,4*num_filters)
        self.maxpool=nn.MaxPool2d(kernel_size=3,stride=2)
        self.dropout=nn.Dropout(.2);self.relu=nn.ReLU();self.flat=nn.Flatten()
        h=size
        for _ in range(3):h=(h-3)//2+1
        self.fc1=nn.Linear(4*num_filters*h*h,num_neurons)
        self.fc2=nn.Linear(num_neurons,num_neurons)
        self.output=nn.Linear(num_neurons,num_classes)
    def forward(self,x):
        x=self.maxpool(self.block_1(x));x=self.maxpool(self.block_2(x));x=self.maxpool(self.block_3(x))
        x=self.flat(self.dropout(x));x=self.relu(self.fc1(x));x=self.relu(self.fc2(x))
        return self.output(x)

class TransferNet(nn.Module):
    """Lab 3 cells 25/34: the same HF ResNet-50 and replaced classifier."""
    def __init__(self,classes,pretrained=True):
        super().__init__()
        from transformers import AutoModelForImageClassification,ResNetConfig,ResNetForImageClassification
        self.net=(AutoModelForImageClassification.from_pretrained('microsoft/resnet-50') if pretrained
                  else ResNetForImageClassification(ResNetConfig()))
        self.net.classifier=nn.Sequential(nn.Flatten(start_dim=1),nn.Linear(self.net.config.hidden_sizes[-1],classes))
        self.net.config.num_labels=classes
    def forward(self,x):return self.net(x).logits
    @property
    def classifier(self):return self.net.classifier

def make_model(regime,classes,pretrained=True,size=224):
    return ConvNN(classes,size) if regime=='scratch' else TransferNet(classes,pretrained)

def freeze_backbone(model,freeze):
    for p in model.net.base_model.parameters():p.requires_grad=not freeze
    for p in model.classifier.parameters():p.requires_grad=True

In [ ]:
classes=['empty','animal']
example_model=make_model(REGIMES[0],len(classes),size=CFG['image_size'])
if REGIMES[0]!='scratch':freeze_backbone(example_model,True)
print(example_model)
print('Trainable parameters:',sum(p.numel() for p in example_model.parameters() if p.requires_grad))
example_model.eval()
with torch.no_grad():print('Forward shape:',example_model(torch.zeros(2,3,CFG['image_size'],CFG['image_size'])).shape)
del example_model


## 4. المقاييس والمعايرة — Labs 2–3 + متطلب المشروع
Top-k = min(3,عدد الفئات)، لذلك في التصنيف الثنائي top-2 ليس مقياسًا مفيدًا للمقارنة. ECE يُحسب من الثقة؛ temperature والعتبة يتعلمان على val_calibrate فقط.

In [ ]:
import numpy as np
from scipy.special import softmax, logsumexp
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score


def ece(y, probabilities, bins=15):
    y=np.asarray(y);p=np.asarray(probabilities)
    if len(y)==0:return None
    conf=p.max(1);correct=p.argmax(1)==y
    indices=np.minimum((conf*bins).astype(int),bins-1)
    return float(sum(np.mean(indices==i)*abs(correct[indices==i].mean()-conf[indices==i].mean())
                     for i in range(bins) if np.any(indices==i)))


def fit_temperature(logits,y):
    logits=np.asarray(logits);y=np.asarray(y,dtype=int)
    if not len(y):raise ValueError('No known-class calibration examples')
    def nll(log_t):
        z=logits/np.exp(log_t)
        return float(np.mean(logsumexp(z,axis=1)-z[np.arange(len(y)),y]))
    result=minimize_scalar(nll,bounds=(np.log(0.25),np.log(10)),method='bounded')
    return float(np.exp(result.x))


def wilson(k,n,z=1.959963984540054):
    if n==0:return [None,None]
    p=k/n;d=1+z*z/n
    centre=(p+z*z/(2*n))/d
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return [float(max(0,centre-half)),float(min(1,centre+half))]


def triage_metrics(y, animal_scores, threshold):
    y=np.asarray(y);pred=np.asarray(animal_scores)>=threshold
    pos=y==1;neg=y==0
    tp=int(np.sum(pred & pos));fn=int(np.sum(~pred & pos));tn=int(np.sum(~pred & neg));fp=int(np.sum(pred & neg))
    return dict(threshold=float(threshold),animal_recall=float(tp/pos.sum()) if pos.any() else None,
                empty_removal_rate=float(tn/neg.sum()) if neg.any() else None,
                animal_frames_lost=fn,animal_frames=int(pos.sum()),empty_frames=int(neg.sum()),
                removed_frames=tn+fn,tp=tp,tn=tn,fp=fp,fn=fn,
                animal_recall_wilson95_independence_assumption=wilson(tp,int(pos.sum())))


def choose_threshold(y, scores, target=0.99):
    if set(np.asarray(y).tolist())!={0,1}:raise ValueError('Calibration needs empty and animal labels')
    candidates=np.unique(np.r_[0.,scores,1.])
    feasible=[triage_metrics(y,scores,t) for t in candidates]
    feasible=[r for r in feasible if r['animal_recall']>=target]
    best=max(feasible,key=lambda r:(r['empty_removal_rate'],r['threshold']))
    best['selection_split']='val_calibrate'
    best['recall_target_is_empirical_not_guaranteed']=True
    return best


def classification_metrics(y,probs,classes):
    y=np.asarray(y);probs=np.asarray(probs);pred=probs.argmax(1);n=len(classes)
    support=np.bincount(y,minlength=n);supported=np.flatnonzero(support)
    report=classification_report(y,pred,labels=list(range(n)),target_names=classes,output_dict=True,zero_division=0)
    k=min(3,n)
    return dict(n=len(y),accuracy=float(accuracy_score(y,pred)),
       macro_f1_all_training_classes=float(f1_score(y,pred,labels=list(range(n)),average='macro',zero_division=0)),
       macro_f1_supported_classes=float(f1_score(y,pred,labels=supported,average='macro',zero_division=0)),
       top_k=k,top_k_accuracy=float(np.mean([label in indices for label,indices in zip(y,np.argsort(probs,axis=1)[:,-k:])])),
       ece=ece(y,probs),per_class=report,confusion_matrix=confusion_matrix(y,pred,labels=list(range(n))).tolist(),
       classes_absent_from_this_split=[classes[i] for i in range(n) if support[i]==0])


def camera_bootstrap(y,probs,cameras,repeats=1000,seed=42,threshold=None):
    """Resample whole test cameras; preserve dependence among frames/crops within camera."""
    y=np.asarray(y);p=np.asarray(probs);c=np.asarray(cameras);sites=np.unique(c)
    if len(sites)<2:return {'status':'not_estimable_fewer_than_two_cameras'}
    rng=np.random.default_rng(seed);acc=[];f1=[];rec=[];empty=[]
    for _ in range(repeats):
        idx=np.concatenate([np.flatnonzero(c==site) for site in rng.choice(sites,len(sites),replace=True)])
        yy=y[idx];pp=p[idx]
        acc.append(float(np.mean(pp.argmax(1)==yy)))
        f1.append(float(f1_score(yy,pp.argmax(1),labels=range(p.shape[1]),average='macro',zero_division=0)))
        if threshold is not None:
            m=triage_metrics(yy,pp[:,1],threshold)
            if m['animal_recall'] is not None:rec.append(m['animal_recall'])
            if m['empty_removal_rate'] is not None:empty.append(m['empty_removal_rate'])
    ci=lambda v:np.quantile(v,[.025,.975]).tolist() if v else [None,None]
    out=dict(unit='camera',cameras=len(sites),repeats=repeats,accuracy95=ci(acc),macro_f1_95=ci(f1),
             caveat='Intervals are descriptive and unstable when there are few held-out cameras.')
    if threshold is not None:out.update(animal_recall95=ci(rec),empty_removal95=ci(empty))
    return out


## 5. حلقة التدريب — forward → loss → backward → step
الكود التالي مرئي بالكامل. الدالة تعيد استخدام نفس الحلقة للمهمتين. لا يدخل Test في التدريب أو اختيار أفضل epoch.

In [ ]:
from scipy.special import softmax
import time,wandb

def loader(samples,train=False):
    gen=torch.Generator().manual_seed(CFG['seed'])
    return DataLoader(FrameDataset(samples,CFG['image_size'],train),batch_size=CFG['batch_size'],
                      shuffle=train,num_workers=0,generator=gen)

def predict(model,data):
    model.eval();all_logits=[];all_y=[]
    with torch.no_grad():
        for images,labels,_ in data:
            all_logits.append(model(images.to(DEVICE)).cpu().numpy());all_y.extend(labels.tolist())
    assert all_logits, 'No known-class samples in this split'
    return np.concatenate(all_logits),np.array(all_y,dtype=int)

def train_model(task,regime):
    seed_everything(CFG['seed'])
    classes=['empty','animal'] if task=='triage' else read_json(DATA/'species.json')
    samples=samples_for_task(rows,task,classes)
    train=[r for r in samples if r['split']=='train' and r['label']>=0]
    val=[r for r in samples if r['split']=='val_select' and r['label']>=0]
    calibrate=[r for r in samples if r['split']=='val_calibrate' and r['label']>=0]
    assert train and val and calibrate
    folder=OUT/task/regime;folder.mkdir(parents=True,exist_ok=True)
    context=artifact_context(CFG,task)
    if (folder/'summary.json').exists():
        saved=read_json(folder/'summary.json');assert saved['context']==context
        assert saved['checkpoint_sha256']==digest(folder/'best.pt')
        print('Reusing completed run:',task,regime);return saved
    assert not (folder/'best.pt').exists(), 'تدريب جزئي موجود؛ استخدم إصدار تجربة جديدًا لتجنب استبداله'
    model=make_model(regime,len(classes),size=CFG['image_size']).to(DEVICE)
    if regime!='scratch':freeze_backbone(model,regime=='frozen')
    criterion=nn.CrossEntropyLoss()
    optimizer=torch.optim.Adam([p for p in model.parameters() if p.requires_grad],lr=CFG['lr_'+regime],weight_decay=CFG['weight_decay'])
    run=wandb.init(project=CFG['project'],name=f'{task}-{regime}',group=DATA_HASH[:12],
                   mode=WANDB_MODE,dir=str(folder.resolve()),config=dict(CFG,task=task,regime=regime),reinit=True)
    best=-1;stale=0;history=[]
    for epoch in range(CFG['epochs']):
        model.train()
        if regime=='frozen':model.eval();model.classifier.train() # Freeze BatchNorm state too.
        total_loss=0;correct=0;total=0
        for images,labels,_ in loader(train,True):
            images,labels=images.to(DEVICE),labels.to(DEVICE)
            optimizer.zero_grad()
            logits=model(images)                     # forward
            loss=criterion(logits,labels)            # compare with true labels
            loss.backward()                         # gradients
            optimizer.step()                        # update weights
            total_loss+=loss.item()*len(labels)
            correct+=int((logits.argmax(1)==labels).sum());total+=len(labels)
        logits,y=predict(model,loader(val))
        m=classification_metrics(y,softmax(logits,axis=1),classes)
        val_loss=float(nn.functional.cross_entropy(torch.tensor(logits),torch.tensor(y)))
        row={'epoch':epoch+1,'train_loss':total_loss/total,'train_accuracy':correct/total,
             'val_loss':val_loss,'val_accuracy':m['accuracy'],'val_macro_f1':m['macro_f1_all_training_classes']}
        history.append(row);run.log(row);print(task,regime,row)
        if row['val_macro_f1']>best:
            best=row['val_macro_f1'];stale=0
            torch.save({'state_dict':model.state_dict(),'regime':regime,'task':task,'classes':classes,
              'image_size':CFG['image_size'],'epoch':epoch+1,'config':CFG,'context':context},folder/'best.pt')
        else:stale+=1
        if stale>=CFG['patience']:break
    pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
    checkpoint=torch.load(folder/'best.pt',map_location=DEVICE,weights_only=False)
    model.load_state_dict(checkpoint['state_dict'])
    z,y=predict(model,loader(calibrate));temperature=fit_temperature(z,y)
    before=softmax(z,axis=1);after=softmax(z/temperature,axis=1)
    policy={'temperature':temperature,'calibration_split':'val_calibrate',
            'species_review_confidence':CFG['species_review_confidence']}
    if task=='triage':
        policy.update(choose_threshold(y,after[:,1],CFG['target_animal_recall']))
        sweep=[triage_metrics(y,after[:,1],t) for t in np.linspace(0,1,101)]
        pd.DataFrame(sweep).to_csv(folder/'validation_threshold_sweep.csv',index=False)
    write_json(folder/'policy.json',policy)
    result={'task':task,'regime':regime,'context':context,'selection_score':best,
      'best_epoch':checkpoint['epoch'],'classes':classes,'checkpoint_sha256':digest(folder/'best.pt'),
      'wandb_run_id':run.id,'wandb_url':run.url if WANDB_MODE=='online' else None,
      'wandb_mode':WANDB_MODE,'calibration_ece_before':ece(y,before),'calibration_ece_after':ece(y,after),
      'train_n':len(train),'val_select_n':len(val),'val_calibrate_n':len(calibrate)}
    run.summary.update({'best_val_macro_f1':best,'calibration_ece_before':ece(y,before),'calibration_ece_after':ece(y,after)})
    run.finish();write_json(folder/'summary.json',result)
    return result


## 6. تدريب فارغة/حيوان

In [ ]:
triage_runs=[train_model('triage',regime) for regime in REGIMES]
display(pd.DataFrame(triage_runs)[['task','regime','selection_score','best_epoch']])

## 7. تدريب الأنواع على القصاصات البشرية

In [ ]:
species_runs=[train_model('species',regime) for regime in REGIMES]
display(pd.DataFrame(species_runs)[['task','regime','selection_score','best_epoch']])

## 8. تثبيت الاختيار قبل Test
يسجل اختيار هذا العضو باستخدام val_select فقط. العضو 4 يقارن اختيارات جميع التجارب من نفس درجات validation.

In [ ]:
selection={task:max([r for r in triage_runs+species_runs if r['task']==task],key=lambda r:r['selection_score'])['regime'] for task in ['triage','species']}
write_json(OUT/'selection.json',{'selected':selection,'criterion':'val_select macro-F1','dataset_hash':DATA_HASH})
print(selection)


## 9. التقييم النهائي وعرض النتائج
المقاييس الأساسية على الأنواع المعروفة مع توضيح coverage والأنواع غير الموجودة في train. فاصل bootstrap حسب الكاميرا يراعي ارتباط الصور لكنه غير مستقر مع عدد قليل من المواقع.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

def evaluate_model(task,regime):
    folder=OUT/task/regime;summary=read_json(folder/'summary.json')
    signature={'data':DATA_HASH,'checkpoint':digest(folder/'best.pt'),'policy':digest(folder/'policy.json')}
    if (folder/'test_metrics.json').exists():
        saved=read_json(folder/'test_metrics.json');assert saved['signature']==signature
        return saved
    checkpoint=torch.load(folder/'best.pt',map_location=DEVICE,weights_only=False)
    assert checkpoint['context']==artifact_context(CFG,task)
    classes=checkpoint['classes'];policy=read_json(folder/'policy.json')
    model=make_model(regime,len(classes),pretrained=False,size=CFG['image_size']).to(DEVICE)
    model.load_state_dict(checkpoint['state_dict'])
    all_test=[r for r in samples_for_task(rows,task,classes) if r['split']=='test']
    known=[r for r in all_test if r['label']>=0];unknown=[r for r in all_test if r['label']<0]
    logits,y=predict(model,loader(known));raw=softmax(logits,axis=1);p=softmax(logits/policy['temperature'],axis=1)
    m=classification_metrics(y,p,classes)
    m.update(task=task,regime=regime,split='test',image_size=CFG['image_size'],policy=policy,
       signature=signature,context=checkpoint['context'],uncalibrated_ece=ece(y,raw),
       known_count=len(known),unknown_count=len(unknown),coverage=len(known)/len(all_test),
       unknown_species=sorted({r['species'] for r in unknown}),
       accuracy_counting_unknown_as_errors=float((p.argmax(1)==y).sum()/len(all_test)),
       accuracy_wilson95_independence_assumption=wilson(int((p.argmax(1)==y).sum()),len(y)),
       camera_bootstrap=camera_bootstrap(y,p,[r['camera'] for r in known],CFG['bootstrap_repeats'],CFG['seed'],
                                      policy['threshold'] if task=='triage' else None))
    if task=='triage':m['operating_point']=triage_metrics(y,p[:,1],policy['threshold'])
    m['per_camera']={camera:classification_metrics(y[idx],p[idx],classes)
      for camera in sorted({r['camera'] for r in known})
      for idx in [[i for i,r in enumerate(known) if r['camera']==camera]]}
    np.savez_compressed(folder/'test_probabilities.npz',y=y,probabilities=p,raw=raw)
    pred=p.argmax(1)
    pd.DataFrame([{'sample_id':r['sample_id'],'camera':r['camera'],'true':classes[y[i]],
      'prediction':classes[pred[i]],'confidence':float(p[i].max())} for i,r in enumerate(known)]).to_csv(folder/'test_predictions.csv',index=False)
    # Triage gallery uses the actual removal threshold, not argmax.
    decisions=(p[:,1]>=policy['threshold']).astype(int) if task=='triage' else pred
    errors=np.flatnonzero(decisions!=y)
    chosen=sorted(errors,key=lambda i:float(p[i].max()),reverse=True)[:12]
    if chosen:
        fig,axes=plt.subplots((len(chosen)+3)//4,4,figsize=(12,3*((len(chosen)+3)//4)))
        for ax in np.asarray(axes).flat:ax.axis('off')
        for ax,i in zip(np.asarray(axes).flat,chosen):
            rgb=cv2.cvtColor(load_rgb_sample(known[i]),cv2.COLOR_BGR2RGB)
            ax.imshow(rgb);ax.set_title(f"{known[i]['sample_id'][:12]}\n{classes[y[i]]} -> {classes[decisions[i]]}",fontsize=8)
        fig.suptitle(f'{task}/{regime}: test errors, input {CFG["image_size"]}, threshold '+str(policy.get('threshold','argmax')))
        fig.tight_layout();fig.savefig(folder/'private_errors.jpg');plt.close(fig)
    run=wandb.init(project=CFG['project'],name=f'{task}-{regime}-test',group=DATA_HASH[:12],
      mode=WANDB_MODE,dir=str(folder.resolve()),config=dict(CFG,task=task,regime=regime,split='locked_test'),reinit=True)
    run.log({'test/accuracy':m['accuracy'],'test/macro_f1':m['macro_f1_all_training_classes'],
             'test/ece':m['ece'],'test/top_k_accuracy':m['top_k_accuracy']})
    run.log({'test/per_class':wandb.Table(dataframe=pd.DataFrame(m['per_class']).T.reset_index())})
    m['wandb_test_run_id']=run.id;m['wandb_test_url']=run.url if WANDB_MODE=='online' else None
    run.finish();write_json(folder/'test_metrics.json',m)
    return m

def show_results(task,regime):
    folder=OUT/task/regime;m=read_json(folder/'test_metrics.json')
    display(pd.DataFrame(m['per_class']).T)
    print('Accuracy:',m['accuracy'],'Top-k:',m['top_k_accuracy'],'ECE:',m['ece'])
    print('Camera bootstrap:',m['camera_bootstrap'],'Unknown crops:',m['unknown_count'])
    h=pd.read_csv(folder/'history.csv');h.plot(x='epoch',y=['train_loss','val_loss'],title=f'{task}/{regime}: train/val_select loss, {CFG["image_size"]}px');plt.tight_layout();plt.savefig(folder/'learning_loss.png');plt.show()
    h.plot(x='epoch',y=['train_accuracy','val_accuracy'],title=f'{task}/{regime}: train/val_select accuracy, argmax');plt.tight_layout();plt.savefig(folder/'learning_accuracy.png');plt.show()
    classes=read_json(folder/'summary.json')['classes']
    ConfusionMatrixDisplay(np.array(m['confusion_matrix']),display_labels=classes).plot(xticks_rotation=90)
    plt.title(f'{task}/{regime}: test, {CFG["image_size"]}px, argmax');plt.tight_layout();plt.savefig(folder/'confusion.png');plt.show()
    z=np.load(folder/'test_probabilities.npz');y=z['y']
    plt.plot([0,1],[0,1],'--',color='gray')
    for key,label in [('raw','Before calibration'),('probabilities','After calibration')]:
        p=z[key];conf=p.max(1);correct=p.argmax(1)==y;bins=np.minimum((conf*10).astype(int),9)
        points=[(conf[bins==b].mean(),correct[bins==b].mean()) for b in range(10) if np.any(bins==b)]
        plt.plot(*zip(*points),'o-',label=label)
    plt.xlabel('Confidence');plt.ylabel('Accuracy');plt.legend();plt.title(f'{task}/{regime}: test reliability, argmax');plt.tight_layout();plt.savefig(folder/'reliability.png');plt.show()
    if (folder/'private_errors.jpg').exists():display(Image.open(folder/'private_errors.jpg'))
    if task=='triage':print('Operating point:',m['operating_point'])


In [ ]:
for task in ['triage','species']:
    for regime in REGIMES:
        evaluate_model(task,regime)
        show_results(task,regime)

## 10. تحليل الأخطاء والمزامنة والتسليم
اكتب IDs وأسبابًا محتملة وخطة بيانات من الأمثلة الفعلية. إذا لم يظهر نوع خطأ، اذكر العدد ولا تختلق أمثلة.

In [ ]:
analysis=OUT/'ERROR_ANALYSIS.md'
if not analysis.exists():analysis.write_text('# Error analysis\n\nTODO: actual sample IDs, rare-class recall/support, false empty removals, calibration, plausible causes, and data improvement plan.\n')
SYNC_WANDB=False
if SYNC_WANDB:
    wandb.login()
    for p in OUT.glob('*/*/wandb/offline-run-*'):
        subprocess.run([sys.executable,'-m','wandb','sync',str(p)],check=True)


In [ ]:
required=[OUT/'config.json',OUT/'selection.json']
for task in ['triage','species']:
    for regime in REGIMES:
        required += [OUT/task/regime/n for n in ['best.pt','summary.json','history.csv','policy.json','test_metrics.json','test_probabilities.npz','test_predictions.csv']]
write_json('handoff/member2_release.json',{'schema':2,'dataset_hash':DATA_HASH,'output_dir':str(OUT),'member':2,'files':{str(p):digest(p) for p in required}})
print('Ready: member2_release.json')


## 11. حفظ نتائجك الخاصة
لا ترفع الصور أو الأوزان إلى GitHub. هذه الحزمة ترسلها للعضو 4، الذي يستورد حزم الأعضاء جميعًا. الدفتر نفسه مستقل.

In [ ]:
import zipfile
archive=ROOT.parent/'Member2_Results.zip'
paths=list(OUT.rglob('*'))+[Path('handoff/member2_release.json'),Path('handoff/requirements_member2.txt'),Path('handoff/runtime_member2.json')]
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        if p.is_file():z.write(p,p.as_posix())
print(archive)
from google.colab import files
files.download(str(archive))
